# P3 scale-aware / severity screening — fixed crop1.5

Attach the existing 00d pooled resources with frozen Train10K/Val3K, CelebA/LCC images and MiniFASNet source/official checkpoint. Edit the paths below. All train/Val/LCC crops are **1.5**.

`RUN_FILTER=None` executes only this notebook's listed variants; a non-empty subset can split variants across Kaggle sessions. Output roots are independent. Every completed run produces its own artifacts/ZIP and root comparison/report/ZIP immediately; resume is independent per run with run/config/geometry checks.

SC scales centers only; SF scales centers and sigma. P3 uses the original selected-worst CE, without harmful gating. No combination or extra search is included.

Optional historical crop15 CSV/JSON summary paths are read only after evaluation. No historical C/P3/R7 control is retrained. No CelebA full Test, detector or alignment. Source Val3K alone selects best/calibrates thresholds; pooled LCC AUC is primary.

Progress is printed for training batches, Val/LCC and export. This single-seed micro round tests hypotheses; it does not declare a final method winner or establish causal proof.


In [ ]:
# Edit paths here. Both notebooks use the same frozen 00d resource bundle.
RESOURCE_ROOT='/kaggle/input/datasets/maithanhphuong/full-resources-v3/minifasnet_binary_crossdomain_resources_v2_lcc_pooled'
CELEBA_ROOT='/kaggle/input/datasets/attentionlayer241/celeba-spoof-for-face-antispoofing/CelebA_Spoof_/CelebA_Spoof'
LCC_ROOT='/kaggle/input/datasets/faber24/lcc-fasd/LCC_FASD'
MINIFASNET_SOURCE_PATH='/kaggle/input/datasets/maithanhphuong/minifasnet/MiniFASNet.py'
MINIFASNET_CHECKPOINT_PATH='/kaggle/input/datasets/maithanhphuong/minifasnet/2.7_80x80_MiniFASNetV2.pth'
OUTPUT_ROOT='/kaggle/working/micro_P3_scaleaware_crop15_v1'
LOG_EVERY_BATCHES=10
RESUME=True
NUM_WORKERS=4
AMP=True
DATA_SEED=43
TRAIN_SEED=100
BATCH_SIZE=256
MAX_EPOCHS=20
MILESTONES=[6,14]
GAMMA=.2
EARLIEST_EARLY_STOP_EPOCH=8
EARLY_STOP_PATIENCE=4
WARMUP_EPOCHS=1
CSMR_PRESERVE_DC=True
SELECTION_CHUNK_SIZE=32
CROP_SCALE=1.5
TRAIN_CROP_FACTOR=1.5
EVAL_CROP_FACTOR=1.5
INPUT_SIZE=80
METRIC_TIE_EPS=1e-12


RUN_FILTER=None  # None = all notebook variants, or a non-empty list of exact IDs.
HISTORICAL_C_CROP15_SUMMARY=None
HISTORICAL_P3_CROP15_SUMMARY=None
HISTORICAL_R7_CROP15_SUMMARY=None
SCALE_RATIO=1.5/2.7
ORIGINAL_CENTERS=[.15,.45,.75]
SCALED_CENTERS=[c*SCALE_RATIO for c in ORIGINAL_CENTERS]
SCALED_SIGMA=.10*SCALE_RATIO
SEVERITY_EPS=1e-8
RUNS={
 'P3_SC_crop15':{'family':'P3','method':'WORST','crop_factor':1.5,'centers':SCALED_CENTERS,'sigma':.10,'clean_ce_weight':.75,'spectral_ce_weight':.25},
 'P3_SF_crop15':{'family':'P3','method':'WORST','crop_factor':1.5,'centers':SCALED_CENTERS,'sigma':SCALED_SIGMA,'clean_ce_weight':.75,'spectral_ce_weight':.25},
}
REPORT_STEM='p3_scaleaware'
def select_runs(run_filter):
    if run_filter is not None and (not isinstance(run_filter,(list,tuple)) or not run_filter or len(set(run_filter))!=len(run_filter) or set(run_filter)-set(RUNS)):
        raise ValueError('RUN_FILTER must be None or a non-empty unique subset of '+', '.join(RUNS))
    return [name for name in RUNS if run_filter is None or name in run_filter]


In [ ]:
PROTOCOL_TEXT="# Scale-aware bands and severity weighting at crop 1.5, protocol v1\n\n## Questions, hypotheses and isolated treatments\n\nAt fixed matched train/evaluation crop 1.5, test whether the historical fixed FFT geometry is misaligned with face-relative structure and whether equally weighting all harmful samples limits R7. Prior reported crop results motivate these questions; they do not prove either mechanism.\n\nThe approximate face-scale ratio is 1.5/2.7 = 0.5555555556. Scaled centers are original [.15,.45,.75] times this ratio: [.0833333333,.25,.4166666667]. Scaled sigma is .10 times this ratio: .0555555556. This approximation uses configured context scale, not per-image boundary-adjusted scale; frozen crop clipping makes it a hypothesis rather than exact physical normalization.\n\nExactly five new runs:\n\n| Notebook | Run | Centers | Sigma | Spectral loss |\n|---|---|---|---|---|\n| 13 | P3_SC_crop15 | scaled | .10 | unconditional selected-worst CE |\n| 13 | P3_SF_crop15 | scaled | .0555555556 | unconditional selected-worst CE |\n| 14 | R7_SC_crop15 | scaled | .10 | original binary harmful CE |\n| 14 | R7_SF_crop15 | scaled | .0555555556 | original binary harmful CE |\n| 14 | R7_SEV_crop15 | original [.15,.45,.75] | .10 | normalized raw-severity CE |\n\nNo fixed-geometry P3/R7/C control is retrained. No SC+SEV/SF+SEV combination, crop/gain/center/sigma/CE-weight search, KL, SmoothL1, margin regression, random/all-band supervision or new run is added.\n\n## Frozen controls and training\n\nReuse notebook 12's matched crop-1.5 micro implementation and notebook 06/07 mechanisms. Read existing mini_train10k.csv and mini_val3k.csv with split seed43/train seed100; no resampling. Preserve binary-v2 MiniFASNetV2/80x80/BGR/[0,1], 2/11/5/40 heads and deterministic official PAD-pretrained initialization. Record identical initialization SHA. Preserve sample/epoch bbox and augmentation seeds plus the independent gain generator.\n\nUse frozen valid raw SCRFD boxes, never detector or GT fallback. Jitter p=.20, scale .95–1.05, shift ±.05 face size; then original crop helper at scale1.5 with its existing boundary scale limit/inward translation. Keep HorizontalFlip .5, ISONoise .2, brightness/contrast limits .2 with p=.3 and MotionBlur limit5 with p=.2. No alignment/normalization/mixup/cutmix/smoothing/weighted sampling. Train, Val and every LCC split use crop1.5.\n\nSGD lr .005/momentum .9/weight decay5e-4, batch256, at most20 epochs, milestones[6,14]/gamma.2 stepped after each epoch. Keep earliest early-stop epoch8/patience4, source-Val3K minimum ACER and higher AUC tie-break; no full-scale phase-aware change. One clean warm-up epoch is excluded from spectral best selection/patience. Only Val3K is evaluated inside epochs.\n\nAuxiliary loss is .1 spoof-type CE + .1 lighting CE + Real-only attribute BCE; all auxiliary supervision is clean only. Warm-up loss is clean CE+aux. Active loss is .75 clean CE+.25 the run's spectral CE+aux.\n\n## Selection and severity definition\n\nReuse the tested spectral generator: one gain per sample Uniform(.65,.90), shared over LOW/MID/HIGH and channels; preserve phase/DC, no amplification. Only radial centers/sigma differ in SC/SF. Build each run's masks from its own config, avoiding global geometry mutation.\n\nWorst-band selection temporarily uses eval/no_grad, compares label-aligned margins (z_real-z_attack for Real, negative for Attack), picks the lowest spectral margin, then restores the previous mode. Selection does not update BatchNorm. The tested micro helper also gathers clean margin; original P3 optimization does not use a harmful gate. Perform differentiable clean and selected-worst forwards only for the loss.\n\nOriginal R7 SC/SF retain detached harmful=(m_worst<m_clean), normalized CE over harmful samples and differentiable zero when none.\n\nFor SEV only:\n\n```\nw_i = max(0, m_clean_i - m_worst_i).detach()\neps = 1e-8\nCE_severity = sum(w_i * CE_worst_i) / (sum(w_i) + eps), if sum(w_i) > eps\nCE_severity = differentiable zero, otherwise\nL_SEV = .75 CE_clean + .25 CE_severity + L_aux\n```\n\nNo gradients flow through selection margins, positive delta or weights. No severity clipping/square/exponent/temperature or tuning; positivity is the specified max(0,delta). SEV retains original centers and sigma to isolate weighting.\n\n## Evaluation and diagnostics\n\nReload best, infer Val3K at crop1.5, derive/freeze its source minimum-ACER threshold, then evaluate physical LCC training/development/evaluation and combined, crop1.5. No CelebA full Test is loaded or inferred; no target fitting. Raw populations8299/2948/7580/18827; pooled metrics are recomputed from concatenated predictions including unscored failures, never averaged. Report AUC/EER/TPR@FPR1%, locked-source APCER/BPCER/ACER/HTER/accuracy, threshold, candidate/scored counts and coverage. Primary endpoint is pooled AUC; source ACER/AUC act as guard.\n\nPer-epoch diagnostics retain clean/worst or harmful CE, total loss, worst LOW/MID/HIGH fractions, selection clean/worst margins, margin drop, flips, gains and harmful count/fraction. SEV additionally records actual epoch-wide mean/median/max positive delta (zero if none), sum of raw weights, positive fraction, effective weighted count (sum(w)^2/sum(w^2)), harmful unweighted mean CE and weighted CE. The latter is the sample-weighted mean of actual per-batch normalized loss, not a new epoch-global training objective. Geometry diagnostics remain descriptive.\n\n## Independence, resume and artifacts\n\nNotebook13 output root: /kaggle/working/micro_P3_scaleaware_crop15_v1/. Notebook14 output root: /kaggle/working/micro_R7_scaleaware_severity_crop15_v1/. Each uses runs/<run_id>/ with config, best, last_state, history, spectral/crop diagnostics, freeze, source/LCC metrics and predictions. No shared writable dependency.\n\nRUN_FILTER=None selects exactly the runs in that notebook; any non-empty unique subset is allowed, unknown/cross-notebook IDs fail. Export root CSV/JSON/report and full root ZIP after every completed selected run, also a standalone run ZIP, so other unfinished variants do not block outputs.\n\nResume restores model/optimizer/scheduler/scaler/epoch, committed best payload, ACER/AUC/patience, histories/geometry and RNG state. One restore_rng definition only; run/config/geometry/crop/init checks prevent cross-variant resume.\n\nOptional HISTORICAL_C_CROP15_SUMMARY / HISTORICAL_P3_CROP15_SUMMARY / HISTORICAL_R7_CROP15_SUMMARY are read-only CSV/JSON summary paths loaded after evaluation. Reports include relevant historical C and own-method crop15 controls and measured deltas; missing references remain pending. Training never depends on them; no old model is trained or inferred.\n\n## Interpretation and validation\n\nAnswer separately: SC versus historical fixed (possible center misalignment), SF versus SC (possible bandwidth role), and fixed-geometry SEV versus historical binary R7 (possible suboptimal equal harmful weighting). SC beating SF may indicate that narrowing removes useful perturbation coverage. HTER-only SEV improvement may primarily reflect calibration; stronger AUC supports separability. Single-seed micro results do not establish causal proof, significance or a final method winner. Do not create a combined geometry+severity experiment now; a future single combination needs independent useful evidence from both questions.\n\nBefore expensive Kaggle training, perform only parse/compile/clear-output checks and tiny offline checks for exact run/filter/geometry definitions, reused preprocessing/model/streams, matched crop, P3/original R7 formulas, detached normalized SEV/zero case, BN safety, source-only calibration and pooled LCC. Static/offline checks do not prove Kaggle CUDA performance or measured research results.\n"

import hashlib, time, importlib.util, inspect, json, math, os, random, zipfile
from collections import Counter, OrderedDict
from pathlib import Path
import albumentations as A
import cv2, numpy as np, pandas as pd, torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.metrics import roc_auc_score, roc_curve
from torch.utils.data import Dataset, DataLoader
from IPython.display import FileLink, display

AUG_SEED=TRAIN_SEED+300000
BBOX_SEED=TRAIN_SEED+500000
GAIN_SEED=TRAIN_SEED+11000
GAIN_RANGE=(.65,.90)
SPLITS=('training','development','evaluation')
EXPECTED_LCC={'training':8299,'development':2948,'evaluation':7580,'combined':18827}
selected=select_runs(RUN_FILTER)
print('[Setup] Selected runs:',selected,flush=True)
if (TRAIN_CROP_FACTOR,EVAL_CROP_FACTOR)!=(1.5,1.5) or any(spec['crop_factor']!=1.5 for spec in RUNS.values()):raise ValueError('This round uses crop 1.5 only')
if NUM_WORKERS<1:raise ValueError('NUM_WORKERS must be >= 1')
if (BATCH_SIZE,MAX_EPOCHS,MILESTONES,GAMMA,TRAIN_SEED)!=(256,20,[6,14],.2,100):raise RuntimeError('Frozen micro recipe changed')
root=Path(RESOURCE_ROOT);out=Path(OUTPUT_ROOT);source=Path(MINIFASNET_SOURCE_PATH);pretrained=Path(MINIFASNET_CHECKPOINT_PATH)
for path in (source,pretrained,root/'manifests/mini_train10k.csv',root/'manifests/mini_val3k.csv',
             root/'bbox/celeba_bbox_cache_full.json',
             root/'labels/celeba_auxiliary_labels.csv'):
    if not path.is_file():raise FileNotFoundError(path)
def read_json(path):return json.loads(Path(path).read_text())
def write_json(path,value):
    path=Path(path);path.parent.mkdir(parents=True,exist_ok=True)
    path.write_text(json.dumps(value,indent=2,sort_keys=True,allow_nan=False)+'\n')
def save_torch(path,value):
    path=Path(path);tmp=path.with_suffix(path.suffix+'.tmp')
    torch.save(value,tmp);os.replace(tmp,path)
def seed_all(seed):
    random.seed(seed);np.random.seed(seed);torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic=True;torch.backends.cudnn.benchmark=False

def state_sha(state):
    h=hashlib.sha256()
    for k in sorted(state):
        v=state[k].detach().cpu().contiguous();h.update(k.encode());h.update(str(v.dtype).encode());h.update(str(tuple(v.shape)).encode());h.update(v.numpy().tobytes())
    return h.hexdigest()

def geometry_summary(g):
    return {'number_of_evaluated_samples':len(g),'boundary_clipped_fraction':float(g[:,0].mean()),
      'mean_crop_width_over_image_width':float(g[:,1].mean()),'mean_crop_height_over_image_height':float(g[:,2].mean()),
      'median_crop_width_over_image_width':float(np.median(g[:,1])),'median_crop_height_over_image_height':float(np.median(g[:,2]))}

def crop_geometry(image,bbox_xyxy,scale=CROP_SCALE,size=INPUT_SIZE):
    # Exact 03b CropImage-compatible geometry.
    src_h,src_w=image.shape[:2];x1,y1,x2,y2=map(float,bbox_xyxy);box_w,box_h=x2-x1,y2-y1
    if box_w<=0 or box_h<=0 or src_w<2 or src_h<2:raise ValueError('Invalid crop geometry')
    requested_w,requested_h=box_w*float(scale),box_h*float(scale)
    requested_cx,requested_cy=(x1+x2)/2,(y1+y2)/2
    limited=(requested_cx-requested_w/2<0 or requested_cy-requested_h/2<0 or requested_cx+requested_w/2>src_w-1 or requested_cy+requested_h/2>src_h-1)
    used=min((src_h-1)/box_h,(src_w-1)/box_w,float(scale));new_w,new_h=box_w*used,box_h*used
    cx,cy=box_w/2+x1,box_h/2+y1;left,top=cx-new_w/2,cy-new_h/2;right,bottom=cx+new_w/2,cy+new_h/2
    if left<0:right-=left;left=0
    if top<0:bottom-=top;top=0
    if right>src_w-1:left-=right-src_w+1;right=src_w-1
    if bottom>src_h-1:top-=bottom-src_h+1;bottom=src_h-1
    lx,ty,rx,by=map(int,(left,top,right,bottom))
    if lx<0 or ty<0 or rx>=src_w or by>=src_h or rx<lx or by<ty:raise RuntimeError('Crop escaped image')
    patch=image[ty:by+1,lx:rx+1]
    if patch.size==0:raise RuntimeError('Empty crop')
    return [float(limited or used<float(scale)),(rx-lx+1)/src_w,(by-ty+1)/src_h]

def require_record(rec,key):
    if rec.get('status')!='VALID' or rec.get('usable_for_pad') is not True or rec.get('bbox_origin')!='SCRFD' or rec.get('source_key')!=key or not str(rec.get('bbox_source','')).startswith('SCRFD'):raise RuntimeError('Unscorable/non-SCRFD: '+key)
    box=rec.get('bbox_xyxy')
    if box is None or len(box)!=4 or 'crop_factor' in rec:raise RuntimeError('Not a raw bbox: '+key)
    x1,y1,x2,y2=map(float,box)
    if not np.isfinite(box).all() or x2<=x1 or y2<=y1:raise RuntimeError('Invalid bbox: '+key)
    return rec

def mini_crop_bgr(image,bbox_xyxy,scale=CROP_SCALE,size=INPUT_SIZE):
    # Exact 03b CropImage-compatible geometry.
    src_h,src_w=image.shape[:2];x1,y1,x2,y2=map(float,bbox_xyxy);box_w,box_h=x2-x1,y2-y1
    if box_w<=0 or box_h<=0 or src_w<2 or src_h<2:raise ValueError('Invalid crop geometry')
    used=min((src_h-1)/box_h,(src_w-1)/box_w,float(scale));new_w,new_h=box_w*used,box_h*used
    cx,cy=box_w/2+x1,box_h/2+y1;left,top=cx-new_w/2,cy-new_h/2;right,bottom=cx+new_w/2,cy+new_h/2
    if left<0:right-=left;left=0
    if top<0:bottom-=top;top=0
    if right>src_w-1:left-=right-src_w+1;right=src_w-1
    if bottom>src_h-1:top-=bottom-src_h+1;bottom=src_h-1
    lx,ty,rx,by=map(int,(left,top,right,bottom))
    if lx<0 or ty<0 or rx>=src_w or by>=src_h or rx<lx or by<ty:raise RuntimeError('Crop escaped image')
    patch=image[ty:by+1,lx:rx+1]
    if patch.size==0:raise RuntimeError('Empty crop')
    return cv2.resize(patch,(size,size))

# The literature's augmentation ranges are fixed. Support both documented Albumentations APIs.
# ISONoise works in RGB/HSV; convert only within augmentation, then restore BGR for MiniFASNet.
iso_params=inspect.signature(A.ISONoise).parameters
brightness_params=inspect.signature(A.RandomBrightnessContrast).parameters
blur_params=inspect.signature(A.MotionBlur).parameters
if {'color_shift','intensity'}<=set(iso_params) and {'brightness_limit','contrast_limit'}<=set(brightness_params) and 'blur_limit' in blur_params:
    AUGMENTATION_API='legacy_limits'
    iso=A.ISONoise(color_shift=(.15,.35),intensity=(.2,.5),p=.2)
    bright=A.RandomBrightnessContrast(brightness_limit=.2,contrast_limit=.2,brightness_by_max=True,p=.3)
    motion=A.MotionBlur(blur_limit=5,p=.2)
elif {'color_shift_range','intensity_range'}<=set(iso_params) and {'brightness_range','contrast_range'}<=set(brightness_params) and 'blur_range' in blur_params:
    AUGMENTATION_API='range_parameters'
    iso=A.ISONoise(color_shift_range=(.15,.35),intensity_range=(.2,.5),p=.2)
    bright=A.RandomBrightnessContrast(brightness_range=(-.2,.2),contrast_range=(-.2,.2),brightness_by_max=True,p=.3)
    motion=A.MotionBlur(blur_range=(3,5),p=.2)
else:raise RuntimeError('Unsupported Albumentations augmentation API; freeze and audit a compatible version')
COPY_AUG=A.Compose([A.HorizontalFlip(p=.5),iso,bright,motion])
def copied_aug_bgr(patch):
    rgb=cv2.cvtColor(patch,cv2.COLOR_BGR2RGB)
    augmented=COPY_AUG(image=rgb)['image']
    return cv2.cvtColor(augmented,cv2.COLOR_RGB2BGR)
def tensor_of(patch):
    if patch.shape!=(80,80,3) or patch.dtype!=np.uint8:raise ValueError('Expected BGR uint8 80x80')
    return torch.from_numpy(np.ascontiguousarray(patch.transpose(2,0,1))).float().div_(255.)

class PADNet(nn.Module):
    def __init__(self,copied):
        super().__init__();self.copied=copied
        self.trunk=module.MiniFASNetV2(embedding_size=128,conv6_kernel=(5,5),drop_p=.2,num_classes=3,img_channel=3)
        if copied:
            self.trunk.prob=nn.Identity()
            self.pad_head=nn.Linear(128,2);self.spoof_type_head=nn.Linear(128,11)
            self.lighting_head=nn.Linear(128,5);self.attributes_head=nn.Linear(128,40)
    def features(self,x):
        t=self.trunk
        for layer in (t.conv1,t.conv2_dw,t.conv_23,t.conv_3,t.conv_34,t.conv_4,t.conv_45,t.conv_5,t.conv_6_sep,t.conv_6_dw,t.conv_6_flatten):x=layer(x)
        if t.embedding_size!=512:x=t.linear(x)
        x=t.drop(t.bn(x))
        if x.ndim!=2 or x.shape[1]!=128:raise RuntimeError('MiniFAS representation is not [B,128]')
        return x
    def forward(self,x,aux=False):
        z=self.features(x)
        if not self.copied:return self.trunk.prob(z)
        pad=self.pad_head(z)
        if aux:return pad,self.spoof_type_head(z),self.lighting_head(z),self.attributes_head(z)
        return pad

def pad_score(logits):
    z=torch.as_tensor(logits)
    if z.shape[1]==3:return (z[:,0]-torch.logsumexp(z[:,1:],dim=1)).float().cpu().numpy()
    if z.shape[1]==2:return (z[:,0]-z[:,1]).float().cpu().numpy()
    raise ValueError('Expected two or three PAD logits')
def threshold_from_val(y3,scores):
    y=np.asarray(y3,dtype=np.int64);s=np.asarray(scores,dtype=np.float64)
    if len(y)!=len(s) or not len(y) or not np.isfinite(s).all() or set(np.unique(y))!={0,1,2}:raise ValueError('Val needs all three classes')
    order=np.argsort(s,kind='mergesort');v=s[order];real=(y[order]==0);nr=int(real.sum());na=len(y)-nr
    bound=np.r_[0,np.flatnonzero(np.diff(v))+1,len(v)];rp=np.r_[0,np.cumsum(real)];ap=np.r_[0,np.cumsum(~real)];pairs=[]
    for j,k in enumerate(bound):
        t=float(np.nextafter(v[0],-np.inf)) if j==0 else (float(np.nextafter(v[-1],np.inf)) if j==len(bound)-1 else float(v[k-1]/2+v[k]/2))
        pairs.append((.5*(rp[k]/nr+(na-ap[k])/na),abs(t),t))
    return min(pairs)[2]
def metrics(y3,scores,threshold,candidate_n):
    y=np.asarray(y3,dtype=int);s=np.asarray(scores,dtype=float);real=y==0
    if len(y)!=len(s) or set(real)!={False,True} or not np.isfinite(s).all():raise ValueError('Invalid metric population')
    fpr,tpr,thr=roc_curve(real.astype(int),s);eer_idx=int(np.argmin(abs(fpr-(1-tpr))))
    pred=s>=threshold;apcer=float(pred[~real].mean());bpcer=float((~pred[real]).mean())
    return {'auc':float(roc_auc_score(real.astype(int),s)),'eer':float(.5*(fpr[eer_idx]+1-tpr[eer_idx])),
      'tpr_at_fpr_1pct':float(np.interp(.01,fpr,tpr)),'apcer':apcer,'bpcer':bpcer,'acer':.5*(apcer+bpcer),
      'hter':.5*(apcer+bpcer),'binary_accuracy':float((pred==real).mean()),'locked_source_threshold':float(threshold),
      'candidate_n':int(candidate_n),'scored_n':len(y),'detector_coverage':len(y)/candidate_n}

def radial_masks(centers,sigma,size=INPUT_SIZE,device=None):
    if device is None:device=globals()['device']
    # Same radial geometry as the tested 03b CSMR notebook.
    y=torch.arange(size,device=device,dtype=torch.float32)-size//2
    yy,xx=torch.meshgrid(y,y,indexing='ij')
    radius=torch.sqrt(xx.square()+yy.square())/math.sqrt(2*(size//2)**2)
    centers=torch.as_tensor(centers,dtype=torch.float32,device=device)
    masks=torch.exp(-((radius[None]-centers[:,None,None])**2)/(2*sigma**2))
    if masks.shape!=(3,size,size) or not torch.isfinite(masks).all():raise RuntimeError('Invalid CSMR radial masks')
    return masks
def spectral_views(x,gains,masks):
    if x.ndim!=4 or x.shape[1:]!=(3,80,80) or not torch.isfinite(x).all():raise ValueError('CSMR input')
    x=x.float();gains=gains.float().reshape(-1)
    if len(gains)!=len(x) or not ((gains>=0)&(gains<=1)).all():raise ValueError('CSMR gain')
    spectrum=torch.fft.fftshift(torch.fft.fft2(x,norm='ortho'),dim=(-2,-1))
    fields=1-(1-gains[:,None,None,None])*masks[None]
    if CSMR_PRESERVE_DC:fields[:,:,40,40]=1
    raw=torch.fft.ifft2(torch.fft.ifftshift(spectrum[:,None]*fields[:,:,None],dim=(-2,-1)),norm='ortho').real
    clips=((raw<0)|(raw>1)).float().mean(dim=(2,3,4));views=raw.clamp(0,1)
    if not torch.isfinite(views).all():raise RuntimeError('Nonfinite CSMR views')
    return views,clips
def aligned_margin(logits,binary):
    d=logits[:,0].float()-logits[:,1].float()
    return torch.where(binary==0,d,-d)

def auxiliary_loss(outputs,binary,spoof,lighting,attrs):
    pad,spoof_logits,light_logits,attr_logits=outputs
    ls=F.cross_entropy(spoof_logits,spoof);ll=F.cross_entropy(light_logits,lighting);mask=binary==0
    la=F.binary_cross_entropy_with_logits(attr_logits[mask],attrs[mask]) if mask.any() else attr_logits.sum()*0
    return ls,ll,la
def bbox_jitter(rec,rng):
    x1,y1,x2,y2=map(float,rec['bbox_xyxy']);w,h=x2-x1,y2-y1
    if rng.random()<.20:
        base=max(w,h);cx,cy=(x1+x2)/2,(y1+y2)/2;scale=rng.uniform(.95,1.05)
        cx+=rng.uniform(-.05,.05)*base;cy+=rng.uniform(-.05,.05)*base;w*=scale;h*=scale
        return [cx-w/2,cy-h/2,cx+w/2,cy+h/2]
    return [x1,y1,x2,y2]

def copied_aug_bgr_seeded(patch,seed):
    # Augmentation RNG cannot consume bbox or spectral RNG state.
    random.seed(seed);np.random.seed(seed)
    if hasattr(COPY_AUG,'set_random_seed'):COPY_AUG.set_random_seed(seed)
    return copied_aug_bgr(patch)

def load_frame(name):return pd.read_csv(root/'manifests'/(name+'.csv'),keep_default_na=False)
train=load_frame('mini_train10k');val=load_frame('mini_val3k')
if (len(train),len(val))!=(10000,3000):raise RuntimeError('Unexpected frozen CelebA populations')
if train.sample_key.duplicated().any() or val.sample_key.duplicated().any() or set(train.sample_key)&set(val.sample_key):
    raise RuntimeError('Train/Val sample overlap')
if set(train.subject_id.astype(str))&set(val.subject_id.astype(str)):raise RuntimeError('Train/Val subject overlap')
for frame in (train,val):
    if 'sampling_seed' in frame and set(frame.sampling_seed)!={DATA_SEED}:raise RuntimeError('Frozen micro seed differs')
    if not frame.usable_for_pad.astype(bool).all() or set(frame.bbox_origin)!={'SCRFD'} or set(frame.three_class_label)!={0,1,2}:
        raise RuntimeError('Frozen mini split is not fully usable')
records=read_json(root/'bbox/celeba_bbox_cache_full.json')['records']
aux=pd.read_csv(root/'labels/celeba_auxiliary_labels.csv',keep_default_na=False).set_index('sample_key',verify_integrity=True)
for frame in (train,val):
    if not set(frame.sample_key)<=set(records) or not set(frame.sample_key)<=set(aux.index):raise RuntimeError('Missing CelebA cache/aux entry')
lcc_frames={};lcc_records={}
for split in ('training','development','evaluation','combined'):
    name='lcc_official_evaluation_full' if split=='evaluation' else 'lcc_'+split+'_full'
    cache_name='lcc_official_evaluation_bbox_cache' if split=='evaluation' else 'lcc_'+split+'_bbox_cache'
    path=root/'manifests'/(name+'.csv');cache=root/'bbox'/(cache_name+'.json')
    if not path.is_file() or not cache.is_file():raise FileNotFoundError((path,cache))
    frame=pd.read_csv(path,keep_default_na=False)
    if len(frame)!=EXPECTED_LCC[split] or frame.path.duplicated().any():raise RuntimeError('Unexpected LCC population: '+split)
    recs=read_json(cache)['records'] if split!='combined' else None
    if recs is not None and set(frame.path)!=set(recs):raise RuntimeError('LCC manifest/cache keys differ: '+split)
    lcc_frames[split]=frame;lcc_records[split]=recs
ordered_pool=pd.concat([lcc_frames[s].assign(lcc_split=s) for s in SPLITS],ignore_index=True)
if not lcc_frames['combined'].equals(ordered_pool):raise RuntimeError('Combined LCC manifest differs from physical splits')
if not torch.cuda.is_available():raise RuntimeError('Kaggle CUDA GPU required for the frozen micro batch size')
source_spec=importlib.util.spec_from_file_location('local_minifasnet_v2',source)
module=importlib.util.module_from_spec(source_spec);source_spec.loader.exec_module(module)
device=torch.device('cuda');use_amp=bool(AMP)
out.mkdir(parents=True,exist_ok=True);(out/'runs').mkdir(exist_ok=True)
(out/'SCALEAWARE_AND_SEVERITY_CROP15_PROTOCOL_v1.md').write_text(PROTOCOL_TEXT)
print('GPU:',torch.cuda.get_device_name(0),'Train/Val:',len(train),len(val),'LCC:',{s:len(f) for s,f in lcc_frames.items()},flush=True)

def init_model():
    seed_all(TRAIN_SEED)
    model=PADNet(True)
    raw=torch.load(pretrained,map_location='cpu',weights_only=True)
    if not isinstance(raw,OrderedDict) or not raw or not all(k.startswith('module.') for k in raw):
        raise RuntimeError('Official PAD checkpoint format differs')
    stripped=OrderedDict((k[7:],v) for k,v in raw.items())
    if tuple(stripped['prob.weight'].shape)!=(3,128) or tuple(stripped['conv_6_dw.conv.weight'].shape[-2:])!=(5,5):
        raise RuntimeError('Official PAD checkpoint architecture differs')
    compatible={k:v for k,v in stripped.items() if k!='prob.weight'}
    target=model.trunk.state_dict()
    if set(compatible)!=set(target) or any(tuple(compatible[k].shape)!=tuple(target[k].shape) for k in compatible):
        raise RuntimeError('PAD pretrained trunk keys differ')
    model.trunk.load_state_dict(compatible,strict=True)
    return model.to(device)

class CelebaData(Dataset):
    def __init__(self,frame,training=False,epoch=0,crop_factor=1.5):
        self.rows=frame.reset_index(drop=True);self.training=training;self.epoch=epoch;self.crop_factor=crop_factor
    def __len__(self):return len(self.rows)
    def __getitem__(self,i):
        row=self.rows.iloc[i];key=str(row.sample_key);rec=require_record(records[key],key)
        image=cv2.imread(str(Path(CELEBA_ROOT)/key),cv2.IMREAD_COLOR)
        if image is None:raise FileNotFoundError(key)
        box=bbox_jitter(rec,random.Random(BBOX_SEED+self.epoch*100000+i)) if self.training else rec['bbox_xyxy']
        patch=mini_crop_bgr(image,box,scale=self.crop_factor)
        geometry=np.asarray(crop_geometry(image,box,scale=self.crop_factor),dtype=np.float64)
        if self.training:patch=copied_aug_bgr_seeded(patch,AUG_SEED+self.epoch*100000+i)
        a=aux.loc[key]
        return tensor_of(patch),int(row.three_class_label),int(row.binary_label),int(a.spoof_type),int(a.lighting),torch.tensor([float(a[f'attr_{j:02d}']) for j in range(40)]),key,geometry

class LCCData(Dataset):
    def __init__(self,split,crop_factor):
        self.split=split;self.crop_factor=crop_factor;self.rows=lcc_frames[split].loc[lcc_frames[split].usable_for_pad.astype(bool)].reset_index(drop=True)
    def __len__(self):return len(self.rows)
    def __getitem__(self,i):
        row=self.rows.iloc[i];key=str(row.path);rec=require_record(lcc_records[self.split][key],key)
        path=(Path(LCC_ROOT)/'LCC_FASD_evaluation'/key) if self.split=='evaluation' else Path(LCC_ROOT)/key
        image=cv2.imread(str(path),cv2.IMREAD_COLOR)
        if image is None:raise FileNotFoundError(path)
        return tensor_of(mini_crop_bgr(image,rec['bbox_xyxy'],scale=self.crop_factor)),int(row.label),key,np.asarray(crop_geometry(image,rec['bbox_xyxy'],scale=self.crop_factor),dtype=np.float64)

def loader(ds,shuffle=False,epoch=0):
    gen=torch.Generator().manual_seed(TRAIN_SEED+epoch)
    return DataLoader(ds,batch_size=BATCH_SIZE,shuffle=shuffle,num_workers=NUM_WORKERS,pin_memory=True,
                      drop_last=False,generator=gen)

def infer(model,dl,lcc=False,label='Evaluation'):
    model.eval();keys=[];ys=[];zs=[];geometry=[];started=time.monotonic()
    print('['+label+'] Starting',len(dl.dataset),'scored images',flush=True)
    with torch.no_grad():
        for step,batch in enumerate(dl,1):
            if lcc:x,y,k,g=batch
            else:x,y,_,_,_,_,k,g=batch
            with torch.cuda.amp.autocast(enabled=use_amp):z=model(x.to(device,non_blocking=True))
            keys.extend(k);ys.extend(y.numpy().tolist());zs.append(z.float().cpu().numpy());geometry.append(g.numpy())
            if step==1 or step%LOG_EVERY_BATCHES==0 or step==len(dl):
                print(f'[{label}] batch {step}/{len(dl)}, elapsed {time.monotonic()-started:.1f}s',flush=True)
    if not zs:raise RuntimeError('No scorable samples')
    return list(map(str,keys)),np.asarray(ys,dtype=int),np.concatenate(zs),np.concatenate(geometry)

def select_worst_view(model,base,binary,masks,gain_rng):
    was_training=model.training;model.eval()
    chosen=[];bands=[];gains_out=[];clips_out=[];clean_margins=[];worst_margins=[]
    try:
        with torch.no_grad():
            all_gains=torch.empty(len(base),device=base.device).uniform_(*GAIN_RANGE,generator=gain_rng)
            for start in range(0,len(base),SELECTION_CHUNK_SIZE):
                x=base[start:start+SELECTION_CHUNK_SIZE];y=binary[start:start+SELECTION_CHUNK_SIZE]
                gains=all_gains[start:start+len(x)]
                with torch.cuda.amp.autocast(enabled=use_amp):clean_z=model(x)
                clean_margins.append(aligned_margin(clean_z,y).detach())
                views,clips=spectral_views(x,gains,masks);margins=[]
                for band in range(3):
                    with torch.cuda.amp.autocast(enabled=use_amp):z=model(views[:,band])
                    margins.append(aligned_margin(z,y))
                margin=torch.stack(margins,dim=1);ix=margin.argmin(dim=1)
                row=torch.arange(len(x),device=x.device)
                chosen.append(views[row,ix].detach());bands.append(ix.detach())
                gains_out.append(gains.detach());clips_out.append(clips[row,ix].detach())
                worst_margins.append(margin[row,ix].detach())
    finally:model.train(was_training)
    return tuple(torch.cat(seq) for seq in (chosen,bands,gains_out,clips_out,clean_margins,worst_margins))


def training_terms(model,x,binary,spoof,lighting,attrs,spec,masks,gain_rng,active):
    method=spec['method'];n=len(x);selected=None;bands=None;gains=None;select_clean=None;select_worst=None
    if active:
        selected,bands,gains,_,select_clean,select_worst=select_worst_view(model,x,binary,masks,gain_rng)
    details={'positive_delta':np.empty(0,dtype=np.float32),'harmful_ce_sum':0.}
    with torch.cuda.amp.autocast(enabled=use_amp):
        clean=model(x,aux=True);clean_ce=F.cross_entropy(clean[0],binary)
        ls,ll,la=auxiliary_loss(clean,binary,spoof,lighting,attrs);aux_total=.1*ls+.1*ll+la
        spectral_ce=clean_ce*0;worst_ce=clean_ce*0;harmful_count=0;spectral_logits=None
        if active:
            spectral_logits=model(selected);ce_each=F.cross_entropy(spectral_logits,binary,reduction='none');worst_ce=ce_each.mean()
            spectral_ce=ce_each.mean()
        loss=.75*clean_ce+.25*spectral_ce+aux_total if active else clean_ce+aux_total
    with torch.no_grad():
        cm=aligned_margin(clean[0],binary).mean()
        flips=int(((clean[0].argmax(1)==binary)&(spectral_logits.argmax(1)!=binary)).sum()) if spectral_logits is not None else 0
        scalars={'train_total_loss':float(loss.detach()),'clean_ce':float(clean_ce.detach()),
          'spectral_ce':float(spectral_ce.detach()),'worst_ce':float(worst_ce.detach()),'aux_loss':float(aux_total.detach()),
          'clean_aligned_margin':float(select_clean.mean()) if select_clean is not None else float(cm),
          'train_clean_aligned_margin':float(cm),'selection_clean_margin':float(select_clean.mean()) if select_clean is not None else 0.,
          'worst_margin':float(select_worst.mean()) if select_worst is not None else 0.,
          'margin_drop':float((select_clean-select_worst).mean()) if select_worst is not None else 0.,
          'spectral_flip_rate':flips/n,'harmful_count':harmful_count,'harmful_fraction':harmful_count/n,
          'harmful_ce':float(spectral_ce.detach()) if method=='HARMFUL' and active else 0.,
          'weighted_ce_severity':float(spectral_ce.detach()) if method=='SEVERITY' and active else 0.,
          'sampled_gain_mean':float(gains.mean()) if gains is not None else 0.}
    counts=Counter(bands.detach().cpu().tolist()) if bands is not None else Counter()
    return loss,scalars,counts,details

def snapshot_rng(gain_rng):
    return {'python':random.getstate(),'numpy':np.random.get_state(),'torch':torch.get_rng_state(),
      'cuda':torch.cuda.get_rng_state_all(),'gain':gain_rng.get_state()}

def restore_rng(state,gain_rng):
    random.setstate(state['python']);np.random.set_state(state['numpy']);torch.set_rng_state(state['torch'])
    torch.cuda.set_rng_state_all(state['cuda']);gain_rng.set_state(state['gain'])

def run_config(name,spec):
    return {'run_id':name,'spec':spec,'resource_root':str(root),'train_manifest':'mini_train10k.csv','val_manifest':'mini_val3k.csv',
      'data_seed':DATA_SEED,'train_seed':TRAIN_SEED,'train_crop_factor':spec['crop_factor'],'eval_crop_factor':spec['crop_factor'],
      'base_order':'torch Generator(TRAIN_SEED+epoch)','bbox_seed':BBOX_SEED,'augmentation_seed':AUG_SEED,'gain_seed':GAIN_SEED,
      'pretrained_checkpoint':str(pretrained),'model_source':str(source),'celeba_root':CELEBA_ROOT,'lcc_root':LCC_ROOT,
      'batch_size':BATCH_SIZE,'max_epochs':MAX_EPOCHS,'milestones':MILESTONES,'gamma':GAMMA,'num_workers':NUM_WORKERS,
      'optimizer':{'name':'SGD','lr':.005,'momentum':.9,'weight_decay':5e-4},
      'warmup_epochs':0 if spec['method']=='CLEAN' else WARMUP_EPOCHS,
      'earliest_early_stop_epoch':EARLIEST_EARLY_STOP_EPOCH,'patience':EARLY_STOP_PATIENCE,
      'spectral':{'centers':spec['centers'],'sigma':spec['sigma'],'gain':[.65,.90],'preserve_dc':CSMR_PRESERVE_DC},
      'severity_epsilon':SEVERITY_EPS if spec['method']=='SEVERITY' else None,'input':'80x80 BGR float [0,1]','augmentation_api':AUGMENTATION_API,'albumentations_version':A.__version__,'amp':use_amp}

def prediction_frame(frame,keys,y,z,threshold,keycol):
    scores=pad_score(z)
    scored=pd.DataFrame({keycol:keys,'pad_logit_score':scores,'pred_real':scores>=threshold,
                         'logit_0':z[:,0],'logit_1':z[:,1]})
    result=frame.merge(scored,on=keycol,how='left',validate='one_to_one',sort=False)
    valid=frame.usable_for_pad.astype(bool)
    if result.loc[valid,'pad_logit_score'].isna().any() or result.loc[~valid,'pad_logit_score'].notna().any():
        raise RuntimeError('Prediction accounting differs from frozen candidates')
    result['locked_threshold']=threshold
    return result

def evaluate_lcc(model,folder,locked,crop_factor,crop_diagnostics):
    scored_predictions=[];all_geometry=[]
    for split in SPLITS:
        frame=lcc_frames[split]
        keys,y,z,g=infer(model,loader(LCCData(split,crop_factor)),True,label=folder.name+' LCC '+split)
        crop_diagnostics['lcc_'+split]=geometry_summary(g);all_geometry.append(g)
        expected=frame.loc[frame.usable_for_pad.astype(bool),'path'].astype(str).tolist()
        if keys!=expected:raise RuntimeError('LCC scored order differs: '+split)
        pred=prediction_frame(frame,keys,y,z,locked,'path').assign(lcc_split=split)
        mm=metrics(y,pad_score(z),locked,len(frame))
        write_json(folder/f'lcc_{split}_metrics.json',mm)
        pred.to_csv(folder/'predictions'/f'lcc_{split}_predictions.csv',index=False)
        scored_predictions.append(pred)
    pooled=pd.concat(scored_predictions,ignore_index=True)
    if not pooled[['path','label','status','usable_for_pad','bbox_origin','lcc_split']].equals(
        lcc_frames['combined'][['path','label','status','usable_for_pad','bbox_origin','lcc_split']]):
        raise RuntimeError('Combined prediction candidates differ from 00d manifest')
    usable=pooled.usable_for_pad.astype(bool)
    mm=metrics(pooled.loc[usable,'label'],pooled.loc[usable,'pad_logit_score'],locked,len(pooled))
    write_json(folder/'lcc_combined_metrics.json',mm)
    pooled.to_csv(folder/'predictions'/'lcc_combined_predictions.csv',index=False)
    crop_diagnostics['lcc_combined']=geometry_summary(np.concatenate(all_geometry))


def run_one(name,spec):
    folder=out/'runs'/name;folder.mkdir(parents=True,exist_ok=True);(folder/'predictions').mkdir(exist_ok=True)
    print('['+name+'] Starting matched crop',spec['crop_factor'],flush=True)
    cfg=run_config(name,spec);cfg_path=folder/'config.json'
    if cfg_path.is_file() and read_json(cfg_path)!=cfg:raise RuntimeError(name+' existing config differs')
    write_json(cfg_path,cfg)
    best_path=folder/'best.pth';last_path=folder/'last_state.pth';done_path=folder/'completion.json'
    if done_path.is_file():
        needed=[folder/(k+'_metrics.json') for k in ('val3k','lcc_training','lcc_development','lcc_evaluation','lcc_combined')]
        if not all(p.is_file() for p in needed) or not best_path.is_file():
            raise RuntimeError(name+' incomplete completion outputs')
        print(name,'completed; skipping',flush=True)
        return
    model=init_model()
    initial_sha=state_sha(model.state_dict())
    write_json(folder/'initialization_audit.json',{'initial_state_sha256':initial_sha,'train_seed':TRAIN_SEED,'pretrained_checkpoint':str(pretrained)})
    optimizer=torch.optim.SGD(model.parameters(),lr=.005,momentum=.9,weight_decay=5e-4)
    scheduler=torch.optim.lr_scheduler.MultiStepLR(optimizer,milestones=MILESTONES,gamma=GAMMA)
    scaler=torch.cuda.amp.GradScaler(enabled=use_amp)
    gain_rng=torch.Generator(device=device).manual_seed(GAIN_SEED)
    best_checkpoint=None;crop_diagnostics={'training_epochs':{}}
    history=[];diagnostics=[];best_acer=float('inf');best_auc=-float('inf');best_epoch=0;no_improve=0;start=1
    if last_path.is_file():
        if not RESUME:raise RuntimeError(name+' has a saved state; enable RESUME or use a new output root')
        state=torch.load(last_path,map_location='cpu',weights_only=False)
        if state.get('run_id')!=name or state.get('config')!=cfg:raise RuntimeError(name+' resume run/config differs')
        if state['initial_state_sha256']!=initial_sha:raise RuntimeError('Resume initial state differs')
        best_checkpoint=state['best_checkpoint'];crop_diagnostics=state['crop_diagnostics']
        if state['best_epoch']:
            if best_checkpoint['run_id']!=name or best_checkpoint['epoch']!=state['best_epoch'] or best_checkpoint['config']!=cfg:raise RuntimeError('Resume committed best differs')
            save_torch(best_path,best_checkpoint)
        model.load_state_dict(state['model']);optimizer.load_state_dict(state['optimizer'])
        scheduler.load_state_dict(state['scheduler']);scaler.load_state_dict(state['scaler'])
        history=state['history'];diagnostics=state['diagnostics'];best_acer=state['best_acer']
        best_auc=state['best_auc'];best_epoch=state['best_epoch'];no_improve=state['no_improve']
        start=state['epoch']+1
        if len(history)!=state['epoch']:raise RuntimeError(name+' resume history length differs')
        restore_rng(state['rng'],gain_rng)
        print(name,'resume at epoch',start,flush=True)
    val_dl=loader(CelebaData(val,crop_factor=spec['crop_factor']))
    masks=radial_masks(spec['centers'],spec['sigma'])
    if start<=MAX_EPOCHS and not (history and history[-1]['stop_reached']):
        for epoch in range(start,MAX_EPOCHS+1):
            warmup=spec['method']!='CLEAN' and epoch<=WARMUP_EPOCHS
            active=not warmup
            model.train();totals=Counter();band_count=Counter();seen=0;harmful_count_total=0;train_geometry=[];positive_deltas=[];harmful_ce_sum=0.
            train_ds=CelebaData(train,training=True,epoch=epoch,crop_factor=spec['crop_factor'])
            train_dl=loader(train_ds,shuffle=True,epoch=epoch);started=time.monotonic()
            print(f'[{name}] epoch {epoch}/{MAX_EPOCHS} starting, phase={"warmup" if warmup else "active"}, lr={optimizer.param_groups[0]["lr"]:g}',flush=True)
            for step,(x,_,binary,spoof,lighting,attrs,_,g) in enumerate(train_dl,1):
                train_geometry.append(g.numpy())
                x=x.to(device,non_blocking=True);binary=binary.to(device);spoof=spoof.to(device)
                lighting=lighting.to(device);attrs=attrs.to(device);optimizer.zero_grad(set_to_none=True)
                loss,scalars,bands,details=training_terms(model,x,binary,spoof,lighting,attrs,
                    spec,masks,gain_rng,active)
                scaler.scale(loss).backward();scaler.step(optimizer);scaler.update()
                count=len(x);seen+=count
                for key,value in scalars.items():totals[key]+=value*count
                band_count.update(bands)
                harmful_count_total+=scalars['harmful_count']
                harmful_ce_sum+=details['harmful_ce_sum']
                if len(details['positive_delta']):positive_deltas.append(details['positive_delta'])
                if step==1 or step%LOG_EVERY_BATCHES==0 or step==len(train_dl):
                    print(f'[{name}] epoch {epoch} batch {step}/{len(train_dl)}, loss={totals["train_total_loss"]/seen:.4f}, elapsed={time.monotonic()-started:.1f}s',flush=True)
            crop_diagnostics['training_epochs'][str(epoch)]=geometry_summary(np.concatenate(train_geometry))
            keys,y,z,g=infer(model,val_dl,label=f'{name} epoch {epoch} Val3K')
            if keys!=val.sample_key.astype(str).tolist():raise RuntimeError('Val order differs')
            scores=pad_score(z);threshold=threshold_from_val(y,scores)
            vm=metrics(y,scores,threshold,len(val))
            improved=active and (vm['acer']<best_acer-METRIC_TIE_EPS or
                (abs(vm['acer']-best_acer)<=METRIC_TIE_EPS and vm['auc']>best_auc+METRIC_TIE_EPS))
            if active:
                if improved:
                    best_acer=vm['acer'];best_auc=vm['auc'];best_epoch=epoch;no_improve=0
                    best_checkpoint={'run_id':name,'epoch':epoch,'config':cfg,
                        'val_acer':best_acer,'val_auc':best_auc,
                        'state_dict':{k:v.detach().cpu().clone() for k,v in model.state_dict().items()}}
                    save_torch(best_path,best_checkpoint)
                else:no_improve+=1
            row={'epoch':epoch,'phase':'warmup' if warmup else 'active','lr':optimizer.param_groups[0]['lr'],
                 'eligible_for_final_checkpoint':active,'val_acer':vm['acer'],'val_auc':vm['auc'],
                 'val_eer':vm['eer'],'val_threshold':threshold,'best_epoch':best_epoch,
                 'no_improve_count':no_improve,'clean_ce_weight':1.0 if warmup else spec['clean_ce_weight'],
                 'spectral_ce_weight':0.0 if warmup else spec['spectral_ce_weight'],
                 'method':'CLEAN_WARMUP' if warmup else spec['method'],
                 **{k:v/seen for k,v in totals.items()},
                 **{f'{label}_count':band_count[i] for i,label in enumerate(('low','mid','high'))},
                 **{f'{label}_fraction':band_count[i]/seen for i,label in enumerate(('low','mid','high'))}}
            row['harmful_count']=harmful_count_total
            row['mean_ce_worst_on_harmful']=harmful_ce_sum/harmful_count_total if harmful_count_total else 0.
            if spec['method']=='SEVERITY':
                delta=np.concatenate(positive_deltas).astype(np.float64) if positive_deltas else np.empty(0)
                weight_sum=float(delta.sum());weight_sq_sum=float(np.square(delta).sum())
                row.update({'mean_positive_delta':float(delta.mean()) if len(delta) else 0.,
                  'median_positive_delta':float(np.median(delta)) if len(delta) else 0.,
                  'max_positive_delta':float(delta.max()) if len(delta) else 0.,'severity_weight_sum':weight_sum,
                  'effective_weighted_sample_count':weight_sum**2/weight_sq_sum if weight_sq_sum>0 else 0.,
                  'fraction_delta_positive':len(delta)/seen})
            diagnostics.append(dict(row))
            stop=epoch==MAX_EPOCHS or (active and epoch>=EARLIEST_EARLY_STOP_EPOCH and no_improve>=EARLY_STOP_PATIENCE)
            row['stop_reached']=stop;history.append(row)
            scheduler.step()
            state={'run_id':name,'config':cfg,'epoch':epoch,
                'model':{k:v.detach().cpu().clone() for k,v in model.state_dict().items()},
                'optimizer':optimizer.state_dict(),'scheduler':scheduler.state_dict(),
                'scaler':scaler.state_dict(),'best_epoch':best_epoch,'best_acer':best_acer,'best_auc':best_auc,
                'no_improve':no_improve,'rng':snapshot_rng(gain_rng),'initial_state_sha256':initial_sha,
                'best_checkpoint':best_checkpoint,'crop_diagnostics':crop_diagnostics,
                'history':history,'diagnostics':diagnostics}
            save_torch(last_path,state)
            pd.DataFrame(history).to_csv(folder/'training_history.csv',index=False)
            pd.DataFrame(diagnostics).to_csv(folder/'spectral_diagnostics.csv',index=False)
            write_json(folder/'crop_diagnostics.json',crop_diagnostics)
            print(name,'epoch',epoch,'Val ACER',vm['acer'],'best',best_epoch,'patience',no_improve,flush=True)
            if stop:break
    if not best_path.is_file() or best_epoch<= (0 if spec['method']=='CLEAN' else WARMUP_EPOCHS):
        raise RuntimeError(name+' has no eligible best checkpoint')
    best=torch.load(best_path,map_location='cpu',weights_only=True)
    if best['run_id']!=name or best['config']!=cfg or best['epoch']!=best_epoch:
        raise RuntimeError(name+' best checkpoint identity differs')
    model.load_state_dict(best['state_dict']);model.eval()
    keys,y,z,g=infer(model,val_dl,label=name+' frozen Val3K')
    crop_diagnostics['val3k']=geometry_summary(g)
    if keys!=val.sample_key.astype(str).tolist():raise RuntimeError('Frozen Val order differs')
    locked=threshold_from_val(y,pad_score(z))
    vm=metrics(y,pad_score(z),locked,len(val))
    if abs(vm['acer']-best['val_acer'])>1e-6 or abs(vm['auc']-best['val_auc'])>1e-6:
        raise RuntimeError('Best checkpoint does not reproduce Val selection metrics')
    write_json(folder/'val3k_metrics.json',vm)
    prediction_frame(val,keys,y,z,locked,'sample_key').to_csv(folder/'predictions'/'val3k_predictions.csv',index=False)
    freeze={'run_id':name,'best_epoch':best_epoch,'locked_threshold':locked,'val3k_metrics':vm,
      'train_crop_factor':spec['crop_factor'],'eval_crop_factor':spec['crop_factor'],'threshold_source':'frozen Val3K at matched crop'}
    freeze_path=folder/'evaluation_freeze.json'
    if freeze_path.is_file() and read_json(freeze_path)!=freeze:raise RuntimeError('Evaluation freeze differs')
    write_json(freeze_path,freeze)
    evaluate_lcc(model,folder,locked,spec['crop_factor'],crop_diagnostics)
    write_json(folder/'crop_diagnostics.json',crop_diagnostics)
    write_json(done_path,{'run_id':name,'best_epoch':best_epoch,'locked_threshold':locked,
                          'crop_factor':spec['crop_factor'],'initial_state_sha256':initial_sha})
    del model
    torch.cuda.empty_cache()


In [ ]:
METRIC_COLUMNS=('val3k_acer','val3k_auc','lcc_evaluation_auc','lcc_evaluation_hter','lcc_combined_auc',
 'lcc_combined_eer','lcc_combined_hter','lcc_combined_tpr_at_fpr_1pct')
DELTA_COLUMNS=('val3k_acer','lcc_evaluation_auc','lcc_combined_auc','lcc_combined_eer','lcc_combined_hter','lcc_combined_tpr_at_fpr_1pct')
FAMILY=next(iter(RUNS.values()))['family']

def historical_control(label,value):
    if value is None:return None,[label+' historical crop15 summary not supplied.']
    path=Path(value)
    if not path.is_file():return None,[label+' historical summary unavailable: '+str(path)]
    try:
        if path.suffix.lower()=='.json':
            payload=read_json(path)
            candidates=payload if isinstance(payload,list) else payload.get('rows',[payload])
        else:candidates=pd.read_csv(path,keep_default_na=False).to_dict('records')
        matches=[r for r in candidates if r.get('run_id',r.get('model'))==label+'_crop15']
        if len(matches)!=1:return None,[label+' historical summary must contain exactly one '+label+'_crop15 row.']
        raw=matches[0]
        if 'crop_factor' in raw and float(raw['crop_factor'])!=1.5:return None,[label+' historical row is not crop 1.5.']
        row={'run_id':label+'_crop15','status':'historical','crop_factor':1.5,'centers':[.15,.45,.75],'sigma':.10,'method':label,'source_summary':str(path)}
        for key,value in raw.items():
            target='val3k_'+key[4:] if key.startswith('val_') else key
            if target.startswith(('val3k_','lcc_')):
                try:number=float(value)
                except (ValueError,TypeError):continue
                if np.isfinite(number):row[target]=number
            elif key=='selected_epoch':row[key]=value
        if not any(k in row for k in METRIC_COLUMNS):return None,[label+' historical row lacks comparable source/LCC metrics.']
        return row,[]
    except (OSError,ValueError,TypeError,KeyError) as exc:
        return None,[label+' historical summary skipped: '+str(exc)]

def metric_table(rows):
    lines=['| Run | Status | Centers | Sigma | '+' | '.join(METRIC_COLUMNS)+' |',
      '|---|---|---|---:|'+'---:|'*len(METRIC_COLUMNS)]
    for row in rows:
        centers=', '.join(f'{c:.10f}' for c in row['centers'])
        lines.append('| '+row['run_id']+' | '+row['status']+' | '+centers+' | '+f'{row["sigma"]:.10f}'+' | '+' | '.join(
          f'{row[k]:.6f}' if isinstance(row.get(k),(int,float)) else 'pending' for k in METRIC_COLUMNS)+' |')
    return lines

def difference_text(left,right,lookup):
    a=lookup.get(left,{});b=lookup.get(right,{})
    deltas={key:a[key]-b[key] for key in DELTA_COLUMNS if isinstance(a.get(key),(int,float)) and isinstance(b.get(key),(int,float))}
    if not deltas:return 'Pending one or both measured result rows.'
    text='; '.join(key+f' {value:+.6f}' for key,value in deltas.items())+'.'
    da=deltas.get('lcc_combined_auc');dh=deltas.get('lcc_combined_hter')
    if da is not None and da>0:text+=' Pooled AUC improves descriptively; inspect the source guard and official evaluation before interpretation.'
    elif dh is not None and dh<0:text+=' HTER improvement without pooled AUC improvement may primarily reflect calibration/operating-point behavior.'
    return text

def export_report():
    rows=[]
    for name,spec in RUNS.items():
        folder=out/'runs'/name
        row={'run_id':name,'status':'pending','crop_factor':1.5,'centers':spec['centers'],'sigma':spec['sigma'],'method':spec['method']}
        if (folder/'completion.json').is_file():
            row['status']='complete';row['selected_epoch']=read_json(folder/'completion.json')['best_epoch']
            for split in ('val3k','lcc_training','lcc_development','lcc_evaluation','lcc_combined'):
                for key,value in read_json(folder/(split+'_metrics.json')).items():row[split+'_'+key]=value
        rows.append(row)
    path=HISTORICAL_P3_CROP15_SUMMARY if FAMILY=='P3' else HISTORICAL_R7_CROP15_SUMMARY
    fixed,notes=historical_control(FAMILY,path)
    baseline_c,c_notes=historical_control('C',HISTORICAL_C_CROP15_SUMMARY);notes+=c_notes
    for row in rows:
        for key in DELTA_COLUMNS:
            row['delta_vs_fixed_'+key]=row[key]-fixed[key] if fixed and key in row and key in fixed else None
            row['delta_vs_C_'+key]=row[key]-baseline_c[key] if baseline_c and key in row and key in baseline_c else None
    if fixed:rows.append(fixed)
    if baseline_c:rows.append(baseline_c)
    write_json(out/(REPORT_STEM+'_comparison.json'),rows)
    pd.DataFrame(rows).to_csv(out/(REPORT_STEM+'_comparison.csv'),index=False)
    lookup={r['run_id']:r for r in rows}
    report=['# '+FAMILY+' geometry / severity screening at crop 1.5','',
      'Frozen Train10K/Val3K, train seed100; identical initialized weights and original micro budget. Crop1.5 for train, source calibration and LCC.',
      'Only Val3K selects best and fits the threshold. LCC combined metrics are recomputed from concatenated predictions.',
      'CelebA full Test is skipped. Historical C/P3/R7 controls are read-only and are never trained or inferred here.',
      'Primary endpoint: LCC Combined AUC; source Val3K ACER/AUC are the source-domain guard.','',*metric_table(rows),'',
      '## A. Center scaling versus fixed geometry','',
      FAMILY+'_SC_crop15 minus historical '+FAMILY+'_crop15: '+difference_text(FAMILY+'_SC_crop15',FAMILY+'_crop15',lookup),
      'If SC improves source-guarded target AUC, center misalignment is a plausible contributor; this is not causal proof.','',
      '## B. Bandwidth scaling beyond SC','',
      FAMILY+'_SF_crop15 minus '+FAMILY+'_SC_crop15: '+difference_text(FAMILY+'_SF_crop15',FAMILY+'_SC_crop15',lookup),
      'SF improving over SC suggests bandwidth normalization may matter; SC improving over SF may mean narrowing removes useful perturbation coverage.','',
      '## C. Harmful weighting','']
    if FAMILY=='R7':
        report+=['R7_SEV_crop15 minus historical R7_crop15: '+difference_text('R7_SEV_crop15','R7_crop15',lookup),
          'SEV retains original centers/sigma. A source-guarded target AUC improvement would suggest equal harmful weighting may be suboptimal; HTER-only change may primarily reflect calibration.']
    else:report.append('This notebook tests geometry only. Severity weighting is investigated independently in notebook14; no severity run is included here.')
    report+=['','## Deltas versus historical fixed geometry','',
      '| Run | '+' | '.join(DELTA_COLUMNS)+' |','|---|'+'---:|'*len(DELTA_COLUMNS)]
    for row in rows:
        if row['run_id'] not in RUNS:continue
        report.append('| '+row['run_id']+' | '+' | '.join(f'{row["delta_vs_fixed_"+key]:+.6f}' if row.get('delta_vs_fixed_'+key) is not None else 'pending' for key in DELTA_COLUMNS)+' |')
        if row['status']=='complete':
            folder=out/'runs'/row['run_id'];diag=pd.read_csv(folder/'spectral_diagnostics.csv')
            active=diag.loc[diag.phase=='active']
            if row['method']=='SEVERITY' and not active.empty:
                final=active.iloc[-1]
                report.append('\nSEV last active epoch diagnostics: '+', '.join(k+f'={final[k]:.6f}' for k in ('mean_positive_delta','median_positive_delta','max_positive_delta','severity_weight_sum','effective_weighted_sample_count','fraction_delta_positive','mean_ce_worst_on_harmful','weighted_ce_severity'))+'.')
            for ref,label in ((fixed,'fixed'),(baseline_c,'C')):
                if ref:
                    for split in ('lcc_evaluation','lcc_combined'):
                        for suffix in ('candidate_n','scored_n'):
                            key=split+'_'+suffix
                            if key in ref and row[key]!=ref[key]:notes.append(row['run_id']+' '+key+' differs from historical '+label+'; interpret comparability cautiously.')
    report+=['','## Limits and next step','',
      'Single-seed micro results and small/isolated differences do not establish significance, causal proof or a final method winner.',
      'The scaled geometry approximates configured crop scale; actual context varies with boundary-limited crops.',
      'No geometry+severity combination is implemented. A future single combination requires useful independent evidence for both geometry and severity.',
      'Missing historical controls or selected runs remain pending. Compare CSVs after independent Kaggle sessions complete.',
      'Severity positive-delta mean/median/max and raw weight sum are epoch-wide; weighted CE is the sample-weighted mean of actual per-batch objectives.',
      '',*notes]
    (out/(REPORT_STEM+'_report.md')).write_text('\n'.join(report)+'\n')
    return rows

def zip_outputs(name):
    folder=out/'runs'/name;run_zip=out.parent/(name+'.zip')
    with zipfile.ZipFile(run_zip,'w',compression=zipfile.ZIP_DEFLATED,allowZip64=True) as archive:
        for file in sorted(folder.rglob('*')):
            if file.is_file():archive.write(file,(Path('runs')/name/file.relative_to(folder)).as_posix())
        archive.write(out/'SCALEAWARE_AND_SEVERITY_CROP15_PROTOCOL_v1.md','SCALEAWARE_AND_SEVERITY_CROP15_PROTOCOL_v1.md')
    root_zip=out.parent/(out.name+'.zip')
    print('[Export] Writing:',root_zip,flush=True)
    with zipfile.ZipFile(root_zip,'w',compression=zipfile.ZIP_DEFLATED,allowZip64=True) as archive:
        for file in sorted(out.rglob('*')):
            if file.is_file():archive.write(file,(Path(out.name)/file.relative_to(out)).as_posix())
    display(FileLink(str(run_zip)));display(FileLink(str(root_zip)))

selected=select_runs(RUN_FILTER)
print('[Runner] Selected:',selected,flush=True)
for name in selected:
    try:run_one(name,RUNS[name])
    except torch.cuda.OutOfMemoryError as exc:
        raise RuntimeError('Frozen batch256 ran out of GPU memory; stop instead of changing the recipe') from exc
    export_report();zip_outputs(name)
print('Finished selected '+FAMILY+' crop1.5 variants:',selected,flush=True)
